![Archive file 19, fixed sources v2](assets/archive-window/file-19.png)

# 19 · Understanding forecast errors and category concentration

The error review strengthens the case for a simpler Action and Massively Multiplayer reference. Action's average improvement is very small, while the leading MMO model changes from fourth to first place across the two cutoffs and underpredicts in 23 of 24 target forecast cases. Damped seasonal smoothing leads at both cutoffs for RPG and Strategy; seasonal naive improves on last value at both cutoffs for Indie.

Game-level checks show how a few titles influence observed category totals. These contributions explain changes in the measured activity, with model residuals and causal explanations kept distinct.

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.investigate_fixed_cohort import run
review = run(ROOT)


![](assets/archive-window/section-divider.png)

## Stability across historical cutoffs

The table retains each leading model's error, rank and improvement over last value. A positive improvement percentage indicates lower MAE. MMO illustrates the risk of selecting from an average: its leading seasonal model is 27.25% worse than last value at the January cutoff and 17.05% better at the February cutoff. RPG's improvement also varies considerably, despite ranking first at both cutoffs.

These are the same two development folds used in Notebook 18. The review adds diagnostic detail, rather than independent validation.

In [2]:
stability = review["leader_fold_stability"]
stability[["category", "model", "origin", "mae", "fold_rank", "skill_vs_last_value_pct"]].assign(
    origin=lambda frame: frame.origin.dt.strftime("%Y-%m")
).round(2).rename(columns={"category":"Category", "model":"Model", "origin":"Cutoff", "mae":"MAE", "fold_rank":"Rank", "skill_vs_last_value_pct":"Improvement vs last value (%)"})


Category,Model,Cutoff,MAE,Rank,Improvement vs last value (%)
Action,Damped trend,2024-01,83958.58,1.0,0.15
Action,Damped trend,2024-02,90864.54,3.0,-0.07
Indie,Seasonal naive,2024-01,30769.94,1.0,58.30
Indie,Seasonal naive,2024-02,35048.04,2.0,29.20
Massively Multiplayer,Damped seasonal,2024-01,46795.83,4.0,-27.25
Massively Multiplayer,Damped seasonal,2024-02,51641.47,1.0,17.05
RPG,Damped seasonal,2024-01,51434.73,1.0,68.47
RPG,Damped seasonal,2024-02,48574.28,1.0,3.13
Strategy,Damped seasonal,2024-01,57003.79,1.0,45.03
Strategy,Damped seasonal,2024-02,44318.53,1.0,40.12


![](assets/archive-window/section-divider.png)

## Sensitivity on the same calendar months

Consecutive cutoffs change the available training history and the forecast leads. To compare the resulting forecasts fairly, this diagnostic matches February–December 2025, the 11 shared target months within leads 12–23. It compares the same observed outcomes at different forecast distances.

For Action's damped trend, moving the cutoff by one month shifts the forecast by roughly 97,107 concurrent players on average. This sensitivity is much larger than its average MAE advantage of roughly 30 over last value. The diagnostic combines the effect of new training information and a one-month shorter horizon; it does not isolate either effect.

In [3]:
sensitivity = review["cutoff_sensitivity"].merge(
    review["leading_models"][["category", "model"]], on=["category", "model"], validate="one_to_one"
)
sensitivity.round(2).rename(columns={"category":"Category", "model":"Model", "shared_months":"Shared months", "average_forecast_shift":"Mean forecast shift", "max_forecast_shift":"Largest shift", "first_cutoff_mae":"January cutoff MAE", "second_cutoff_mae":"February cutoff MAE"})


Category,Model,Shared months,Mean forecast shift,Largest shift,January cutoff MAE,February cutoff MAE
Action,Damped trend,11,97106.94,97106.94,89455.32,72513.28
Indie,Seasonal naive,11,5015.44,55169.83,29635.62,34636.38
Massively Multiplayer,Damped seasonal,11,8195.28,18059.73,48635.12,56251.39
RPG,Damped seasonal,11,17448.41,23001.94,49972.02,52497.46
Strategy,Damped seasonal,11,31393.61,33591.35,56012.59,43496.77


![](assets/archive-window/section-divider.png)

## Bias and the largest misses

Positive error means observed activity exceeded the forecast. The bias counts below refer to 24 forecast cases per category: 12 target months at each origin, including repeated calendar months. MMO's leading model underpredicts in 23 cases, which deserves attention even though it has the lowest average category MAE. Action's near-zero mean error combines underprediction and overprediction.

The largest cases are review points, with no observations removed or replaced. Their values have been reconciled with the fixed database. A large residual alone provides insufficient evidence of a source-data error.

In [4]:
review["leader_bias"].round(2).rename(columns={"category":"Category", "model":"Model", "underpredicted_cases":"Underprediction cases", "overpredicted_cases":"Overprediction cases", "mean_error":"Mean error"})


Category,Model,Underprediction cases,Overprediction cases,Mean error
Action,Damped trend,11,13,230.03
Indie,Seasonal naive,11,13,-6619.26
Massively Multiplayer,Damped seasonal,23,1,48953.26
RPG,Damped seasonal,14,10,-19770.80
Strategy,Damped seasonal,7,17,-30304.13


In [5]:
largest = review["largest_leader_errors"].groupby("category", sort=False).head(1)
largest[["case_id", "category", "model", "origin", "month", "actual", "prediction", "error"]].assign(
    origin=lambda frame: frame.origin.dt.strftime("%Y-%m"), month=lambda frame: frame.month.dt.strftime("%Y-%m")
).round(2)


case_id,category,model,origin,month,actual,prediction,error
1,Action,Damped trend,2024-02,2026-01,3396661.79,3103933.47,292728.32
4,RPG,Damped seasonal,2024-01,2025-09,710654.68,862662.99,-152008.31
6,Massively Multiplayer,Damped seasonal,2024-02,2025-03,841758.84,714712.59,127046.25
9,Strategy,Damped seasonal,2024-01,2025-04,978732.58,1092547.95,-113815.37
13,Indie,Seasonal naive,2024-02,2025-04,742288.54,811798.40,-69509.86


![](assets/archive-window/section-divider.png)

## Which games contribute to the observed changes?

For Action's January 2026 case, Counter-Strike 2 records about 283,813 more concurrent players than at the February 2024 origin, while Dota 2 adds about 141,402. Other games partly offset these changes. Summing every game's change exactly reproduces the last-value baseline error for that case.

Seasonal naive requires a different reference. Beyond 12 leads, it continues repeating the final training-year cycle. For the April 2025 Indie miss at the February 2024 cutoff, the reference is April 2023. Path of Exile contributes about −63,251 and Unturned about −40,212 to that seasonal residual, partly offset by Last Epoch at +27,735. The complete per-game contributions sum to the observed seasonal error of −69,510.

The fitted smoothing forecasts have no separate prediction for each game, so the observed app changes are context for their misses. Year-over-year changes are retained separately and use the actual previous calendar year.

In [6]:
indie_case = largest.loc[largest.category.eq("Indie"), "case_id"].iloc[0]
indie_parts = review["top_seasonal_changes"].loc[lambda frame: frame.case_id.eq(indie_case)]
indie_parts[["name", "seasonal_reference_month", "seasonal_reference_activity", "target_activity", "seasonal_residual_contribution"]].assign(
    seasonal_reference_month=lambda frame: frame.seasonal_reference_month.dt.strftime("%Y-%m")
).round(2).rename(columns={"name":"Game", "seasonal_reference_month":"Reference month", "seasonal_reference_activity":"Reference activity", "target_activity":"Observed activity", "seasonal_residual_contribution":"Contribution to error"})


![](assets/archive-window/section-divider.png)

## Concentration within categories

At the January 2026 cutoff, Dota 2 accounts for 46.40% of Strategy activity, and the five largest MMO games account for 69.57% of that category. A category forecast can therefore be sensitive to a small number of titles. These percentages describe the covered cohort, whose labels overlap; they should be read separately for each category.

In [7]:
latest_concentration = review["category_concentration"].sort_values("month").groupby("category").tail(1)
latest_concentration[["category", "top_game", "top1_share_pct", "top5_share_pct"]].round(2).rename(
    columns={"category":"Category", "top_game":"Largest game · January 2026", "top1_share_pct":"Largest game (%)", "top5_share_pct":"Top five games (%)"}
)


Category,Largest game · January 2026,Largest game (%),Top five games (%)
RPG,Rust,12.39,39.81
Indie,Rust,13.83,35.69
Action,Counter-Strike 2,31.51,62.93
Massively Multiplayer,PUBG: BATTLEGROUNDS,36.28,69.57
Strategy,Dota 2,46.40,60.07


![](assets/archive-window/section-divider.png)

## Review outcome

The evidence supports keeping last value as the practical reference for Action and Massively Multiplayer. Seasonal naive remains the Indie candidate, while damped seasonal smoothing remains the RPG and Strategy candidate. This formed the shortlist for the provisional policy recorded in Notebook 20, with no additional tuning or new data sources introduced.

The principal limits remain the two overlapping cutoffs, short training histories and retrospective cohort selection. Notebook 20 presents that forecast, its reference comparison and the historical-error sensitivity ranges.